# 05 · Extension: from location to demand through the price premium

Notebook 04 shows that location does not predict demand directly. Hypothesis: location matters **relative to price**.

1. **Stage 1** (June 2026 snapshot): predict a benchmark log price from listing attributes + all external data, out of fold by host.
2. **Premium** = log(actual quoted price) − benchmark. Positive = priced above comparable listings in a comparable area.
3. **Stage 2**: forecast July reviews and the first review of new listings, comparing history-only models with models that add raw price or the premium.

This is a predictive association, not a causal effect of price.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
from config import CLASSMATE_REPO, SLIM, EXT, OUT
from common import *
from external import build
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
st = pd.read_csv(CLASSMATE_REPO / "data" / "stations_used.csv")   # official IDFM stations (classmate's repo)
print("classmate repo:", CLASSMATE_REPO.exists(), "| slim:", SLIM.exists(), "| external:", EXT.exists())

classmate repo: True | slim: True | external: True


In [2]:
tp = pd.read_csv(CLASSMATE_REPO / "data" / "insideairbnb" / "temporal_review_panel.csv.gz", low_memory=False); J = tp[tp.split == "test"]
sp = pd.read_csv(CLASSMATE_REPO / "data" / "insideairbnb" / "listing_snapshot_panel.csv.gz", low_memory=False); sp = sp[sp.snapshot == "2026-06-16"]
sp["price_eur"] = pd.to_numeric(sp.price.astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
d = J.merge(sp[["id", "price_eur", "property_type"]], on="id", how="inner")
d = d[(d.room_type == "Entire home/apt") & d.price_eur.between(20, 1000) & (d.minimum_nights < 30)].reset_index(drop=True)
X, G, info, _ = build(d.latitude.values, d.longitude.values, EXT, st); d = pd.concat([d, X], axis=1)
d["cell"] = cell_1km(d.latitude, d.longitude); d["block"] = block_3km(d.latitude, d.longitude); cat(d, ["property_type", "room_type"])
EXT_ALL = sum(G.values(), []); print(len(d), "priced entire homes in June")

35133 priced entire homes in June


In [3]:
A1 = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "property_type"]
lp = np.log(d.price_eur.values); rows = []
bench, _ = oof(d, A1 + EXT_ALL, lp, "reg", "host")
d["log_price"] = lp; d["premium"] = lp - bench
rows.append({"task": "T9 stage 1 benchmark (June)", "protocol": "host", "features": "A1+all external", "n": len(d), **score(lp, bench, "reg")})
rows[-1]

{'task': 'T9 stage 1 benchmark (June)',
 'protocol': 'host',
 'features': 'A1+all external',
 'n': 35133,
 'R2': 0.5552229548712371,
 'MAE': 0.28408490690092036}

In [4]:
PROP = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "capture_age_days"]
HIST = ["reviews_30", "reviews_90", "reviews_365", "reviews_total", "days_since_last_review", "days_since_first_review", "superhost", "review_scores_rating"]
obs = d[d.followup_observed].reset_index(drop=True); H = PROP + HIST
S2 = {"A history+property": H, "B +all external": H + EXT_ALL, "P1 +log price": H + ["log_price"], "P2 +premium": H + ["premium"],
      "P3 +log price+all external": H + ["log_price"] + EXT_ALL}
run_task("T9 July reviews, priced homes", obs, obs.target.values.astype(float), "poisson", S2, rows)

T9 July reviews, priced homes host     A history+property                 D2_poisson=0.4719  MAE=0.8696  Spearman=0.6651


T9 July reviews, priced homes host     B +all external                    D2_poisson=0.4688  MAE=0.8723  Spearman=0.6640


T9 July reviews, priced homes host     P1 +log price                      D2_poisson=0.4834  MAE=0.8604  Spearman=0.6712


T9 July reviews, priced homes host     P2 +premium                        D2_poisson=0.4903  MAE=0.8563  Spearman=0.6745


T9 July reviews, priced homes host     P3 +log price+all external         D2_poisson=0.4826  MAE=0.8610  Spearman=0.6719


T9 July reviews, priced homes spatial  A history+property                 D2_poisson=0.4707  MAE=0.8670  Spearman=0.6646


T9 July reviews, priced homes spatial  B +all external                    D2_poisson=0.4647  MAE=0.8721  Spearman=0.6619


T9 July reviews, priced homes spatial  P1 +log price                      D2_poisson=0.4804  MAE=0.8587  Spearman=0.6705


T9 July reviews, priced homes spatial  P2 +premium                        D2_poisson=0.4886  MAE=0.8532  Spearman=0.6743


T9 July reviews, priced homes spatial  P3 +log price+all external         D2_poisson=0.4766  MAE=0.8621  Spearman=0.6687


{('host',
  'A history+property'): array([0.64526582, 1.56500711, 0.89194924, ..., 0.41605213, 0.49138846,
        0.44210893], shape=(31655,)),
 ('host',
  'B +all external'): array([0.61884274, 1.61641847, 0.87929335, ..., 0.61308627, 0.3769456 ,
        0.71347118], shape=(31655,)),
 ('host',
  'P1 +log price'): array([0.5613541 , 1.58563723, 1.01392897, ..., 0.26053001, 0.84713465,
        0.26831709], shape=(31655,)),
 ('host',
  'P2 +premium'): array([0.49160506, 1.45746334, 1.06755007, ..., 0.04310174, 0.93148716,
        0.06755964], shape=(31655,)),
 ('host',
  'P3 +log price+all external'): array([0.5393046 , 1.57896638, 0.97191405, ..., 0.46748367, 0.62888323,
        0.17768782], shape=(31655,)),
 ('spatial',
  'A history+property'): array([0.61408688, 1.79979519, 0.8111371 , ..., 0.47364161, 0.37590403,
        0.34112513], shape=(31655,)),
 ('spatial',
  'B +all external'): array([0.6493949 , 1.66090952, 0.81986856, ..., 0.50205478, 0.20579269,
        0.31264822], shape=

**Descriptive check**: July reviews by premium quintile, among listings active before June. The ratio column divides July reviews by the listing's own pre-June pace, to separate the premium from listing quality.

In [5]:
act = obs[obs.reviews_90 > 0].copy()
act["quintile"] = pd.qcut(act.premium, 5, labels=["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 most expensive"])
tab = act.groupby("quintile", observed=True).agg(n=("id", "size"), premium_median=("premium", "median"), july_reviews_mean=("target", "mean"),
                                                 prior_90d_reviews_mean=("reviews_90", "mean")).reset_index()
tab["july_per_prior_rate"] = tab.july_reviews_mean / (tab.prior_90d_reviews_mean * 31 / 90)
tab.to_csv(OUT / "t9_premium_quintiles.csv", index=False); tab.round(3)

,quintile,n,premium_median,july_reviews_mean,prior_90d_reviews_mean,july_per_prior_rate
0,Q1 cheapest,4038,-0.413,2.171,6.288,1.002
1,Q2,4037,-0.201,1.876,5.817,0.937
2,Q3,4037,-0.047,1.679,5.626,0.867
3,Q4,4037,0.110,1.684,5.334,0.917
4,Q5 most expensive,4037,0.360,1.503,4.830,0.903


In [6]:
cold = obs[obs.reviews_total == 0].reset_index(drop=True)
Rv = pd.read_csv(SLIM / "reviews_slim.csv.gz", usecols=["listing_id", "date"], parse_dates=["date"])
r = Rv.merge(cold[["id", "last_scraped"]].assign(last_scraped=lambda x: pd.to_datetime(x.last_scraped)), left_on="listing_id", right_on="id")
y = (cold.id.map(r[r.date > r.last_scraped].groupby("listing_id").size()).fillna(0) > 0).astype(int).values
print("new priced listings:", len(cold), "| share with a first review:", y.mean().round(3))
S3 = {"A property": PROP, "B +all external": PROP + EXT_ALL, "P1 +log price": PROP + ["log_price"], "P2 +premium": PROP + ["premium"],
      "P3 +log price+all external": PROP + ["log_price"] + EXT_ALL}
run_task("T9b new listing first review", cold, y, "clf", S3, rows)

new priced listings: 3865 | share with a first review: 0.211


T9b new listing first review host     A property                         AUC=0.6657  PR_AUC=0.3523


T9b new listing first review host     B +all external                    AUC=0.6569  PR_AUC=0.3292


T9b new listing first review host     P1 +log price                      AUC=0.7754  PR_AUC=0.4562


T9b new listing first review host     P2 +premium                        AUC=0.7813  PR_AUC=0.4412


T9b new listing first review host     P3 +log price+all external         AUC=0.7819  PR_AUC=0.4487


... (2 outputs left out)

T9b new listing first review spatial  P1 +log price                      AUC=0.7696  PR_AUC=0.4387


T9b new listing first review spatial  P2 +premium                        AUC=0.7809  PR_AUC=0.4374


T9b new listing first review spatial  P3 +log price+all external         AUC=0.7780  PR_AUC=0.4351


{('host',
  'A property'): array([0.08267156, 0.01764829, 0.03214976, ..., 0.17486635, 0.28131677,
        0.20075679], shape=(3865,)),
 ('host',
  'B +all external'): array([0.22084704, 0.0074793 , 0.02777348, ..., 0.21477725, 0.08305152,
        0.16976506], shape=(3865,)),
 ('host',
  'P1 +log price'): array([0.04115009, 0.00391301, 0.16701215, ..., 0.18890742, 0.59766751,
        0.17836197], shape=(3865,)),
 ('host',
  'P2 +premium'): array([0.09527531, 0.00270042, 0.08352408, ..., 0.05609819, 0.55916055,
        0.07822327], shape=(3865,)),
 ('host',
  'P3 +log price+all external'): array([0.0354888 , 0.00372474, 0.09060109, ..., 0.28429347, 0.27812299,
        0.04257914], shape=(3865,)),
 ('spatial',
  'A property'): array([0.11562449, 0.01820648, 0.02975024, ..., 0.30470188, 0.27311592,
        0.12868662], shape=(3865,)),
 ('spatial',
  'B +all external'): array([0.3505177 , 0.00898962, 0.03786129, ..., 0.24518266, 0.09959073,
        0.06796411], shape=(3865,)),
 ('spatial',

In [7]:
t9 = pd.DataFrame(rows); t9.to_csv(OUT / "t9_premium.csv", index=False)
t9["value"] = t9.apply(lambda r: r.get("D2_poisson") if r.task.startswith("T9 July") else r.get("AUC"), axis=1)
t9.dropna(subset=["value"]).pivot_table(index=["task", "features"], columns="protocol", values="value", sort=False).round(3)

protocol                                                   host  spatial
task                          features                                  
T9 July reviews, priced homes A history+property          0.472    0.471
                              B +all external             0.469    0.465
                              P1 +log price               0.483    0.480
                              P2 +premium                 0.490    0.489
                              P3 +log price+all external  0.483    0.477
T9b new listing first review  B +all external             0.657    0.643
                              P1 +log price               0.775    0.770
                              P2 +premium                 0.781    0.781
                              P3 +log price+all external  0.782    0.778
                              A property                  0.666    0.672